In [ ]:
!git clone https://github.com/Ish2905/Comemo-Dataset.git

In [ ]:
%cd /content/Comemo-Dataset

In [ ]:
!git pull

In [ ]:
# ==== DO NOT MODIFY THIS CELL ====
from google.colab import drive
drive.mount('/content/drive')

import duckdb
import os

DB_PATH = "/content/drive/MyDrive/Capstone/comemo.db"

# HARD FAIL if Drive is not mounted
assert os.path.exists("/content/drive/MyDrive"), "Drive not mounted!"

# HARD FAIL if DB file missing (after first creation)
if not os.path.exists(DB_PATH):
    print("⚠️ comemo.db not found yet (first run only)")
else:
    print("✅ Using existing database:", DB_PATH)

con = duckdb.connect(DB_PATH)

# Sanity check
print(con.execute("SHOW TABLES").fetchdf())
# =================================


In [ ]:
con.execute("""COPY (
    WITH base AS (
        SELECT
            parent_asin,
            rating,
            DATE_TRUNC(
                'month',
                to_timestamp(timestamp / 1000)
            ) AS month_start
        FROM reviews_raw
    ),
    first_review AS (
        SELECT
            parent_asin,
            MIN(month_start) AS first_month
        FROM base
        GROUP BY parent_asin
    )
    SELECT
        b.parent_asin,
        EXTRACT(year FROM b.month_start) AS year,
        EXTRACT(month FROM b.month_start) AS month,
        STRFTIME(b.month_start, '%Y-%m') AS year_month,
        COUNT(*) AS monthly_review_count,
        AVG(b.rating) AS monthly_rating_avg,
        DATE_DIFF('month', f.first_month, b.month_start) AS product_age_months
    FROM base b
    JOIN first_review f
        ON b.parent_asin = f.parent_asin
    GROUP BY
        b.parent_asin,
        year,
        month,
        year_month,
        b.month_start,
        f.first_month
)
TO '/content/drive/MyDrive/Capstone/reviews_monthly.parquet'
(FORMAT PARQUET);""")


In [ ]:
con.execute("""
SELECT *
FROM read_parquet('/content/drive/MyDrive/Capstone/reviews_monthly.parquet')
LIMIT 10;
""").fetchdf()


In [ ]:
con.execute("""COPY (
    SELECT
        *,
        (monthly_review_count -
         LAG(monthly_review_count) OVER (
            PARTITION BY parent_asin
            ORDER BY year_month
         )) /
        NULLIF(
            LAG(monthly_review_count) OVER (
                PARTITION BY parent_asin
                ORDER BY year_month
            ), 0
        ) AS review_growth_rate,

        monthly_review_count / 30.0 AS review_velocity,

        monthly_rating_avg -
        LAG(monthly_rating_avg) OVER (
            PARTITION BY parent_asin
            ORDER BY year_month
        ) AS monthly_rating_change

    FROM read_parquet(
        '/content/drive/MyDrive/Capstone/reviews_monthly.parquet'
    )
)
TO '/content/drive/MyDrive/Capstone/reviews_momentum.parquet'
(FORMAT PARQUET);
""")


In [ ]:
con.execute("""
SELECT *
FROM read_parquet('/content/drive/MyDrive/Capstone/reviews_momentum.parquet')
LIMIT 10;
""").fetchdf()

In [ ]:
con.execute("""
SELECT
    monthly_review_count,
    COUNT(*) AS freq
FROM read_parquet(
    '/content/drive/MyDrive/Capstone/reviews_monthly.parquet'
)
GROUP BY monthly_review_count
ORDER BY monthly_review_count;
""").fetchdf()


In [ ]:
con.execute("""SELECT *
FROM read_parquet(
    '/content/drive/MyDrive/Capstone/reviews_monthly.parquet'
)
WHERE monthly_review_count > 20
ORDER BY monthly_review_count DESC
LIMIT 10;""").fetchdf()


In [ ]:
con.execute("""COPY (
    WITH trust_base AS (
        SELECT
            parent_asin,
            STRFTIME(
                DATE_TRUNC('month', to_timestamp(timestamp / 1000)),
                '%Y-%m'
            ) AS year_month,
            AVG(CAST(verified_purchase AS INT)) AS verified_ratio,
            SUM(helpful_vote) * 1.0 / COUNT(*) AS engagement_score
        FROM reviews_raw
        GROUP BY parent_asin, year_month
    )
    SELECT
        m.*,
        t.verified_ratio,
        t.engagement_score
    FROM read_parquet(
        '/content/drive/MyDrive/Capstone/reviews_momentum.parquet'
    ) m
    LEFT JOIN trust_base t
        ON m.parent_asin = t.parent_asin
       AND m.year_month = t.year_month
)
TO '/content/drive/MyDrive/Capstone/reviews_trust.parquet'
(FORMAT PARQUET);""")


In [ ]:
con.execute("""
SELECT *
FROM read_parquet('/content/drive/MyDrive/Capstone/reviews_trust.parquet')
LIMIT 10;
""").fetchdf()

In [ ]:
con.execute("""SELECT *
FROM read_parquet(
    '/content/drive/MyDrive/Capstone/reviews_trust.parquet'
)
WHERE monthly_review_count > 20
ORDER BY monthly_review_count DESC
LIMIT 10;""").fetchdf()

In [ ]:
con.execute("""COPY (
    WITH category_avg AS (
        SELECT
            m.main_category,
            AVG(r.review_growth_rate) AS category_avg_growth
        FROM read_parquet(
            '/content/drive/MyDrive/Capstone/reviews_trust.parquet'
        ) r
        JOIN metadata_raw m
            ON r.parent_asin = m.parent_asin
        GROUP BY m.main_category
    )
    SELECT
        r.*,
        m.main_category,
        c.category_avg_growth,
        r.review_growth_rate - c.category_avg_growth
            AS relative_trend_score
    FROM read_parquet(
        '/content/drive/MyDrive/Capstone/reviews_trust.parquet'
    ) r
    JOIN metadata_raw m
        ON r.parent_asin = m.parent_asin
    JOIN category_avg c
        ON m.main_category = c.main_category
)
TO '/content/drive/MyDrive/Capstone/reviews_category.parquet'
(FORMAT PARQUET);""")


In [ ]:
con.execute("""
SELECT
    main_category,
    AVG(relative_trend_score) AS avg_rel_trend
FROM read_parquet(
    '/content/drive/MyDrive/Capstone/reviews_category.parquet'
)
GROUP BY main_category
ORDER BY avg_rel_trend DESC
LIMIT 10;
""").fetchdf()


In [ ]:
con.execute("""
SELECT
    main_category,
    COUNT(*) AS row_count
FROM read_parquet(
    '/content/drive/MyDrive/Capstone/reviews_category.parquet'
)
GROUP BY main_category
ORDER BY row_count DESC;""").fetchdf()


In [ ]:
con.execute("""SELECT *
FROM read_parquet(
    '/content/drive/MyDrive/Capstone/reviews_category.parquet'
)
WHERE monthly_review_count > 20
ORDER BY monthly_review_count DESC
LIMIT 10;""").fetchdf()

In [ ]:
con.execute("""COPY (
    SELECT
        *,
        CASE
            WHEN product_age_months <= 6
                 AND relative_trend_score > 0
            THEN 'emerging'

            WHEN relative_trend_score > 0.2
                 AND review_growth_rate > 0
            THEN 'growing'

            WHEN monthly_review_count >= 20
                 AND relative_trend_score BETWEEN -0.05 AND 0.2
            THEN 'peaking'

            WHEN ABS(relative_trend_score) < 0.05
            THEN 'stable'

            ELSE 'declining'
        END AS trend_lifecycle_stage
    FROM read_parquet(
        '/content/drive/MyDrive/Capstone/reviews_category.parquet'
    )
)
TO '/content/drive/MyDrive/Capstone/reviews_lifecycle.parquet'
(FORMAT PARQUET);
""")

In [ ]:
con.execute("""
SELECT *
FROM read_parquet(
  '/content/drive/MyDrive/Capstone/reviews_lifecycle.parquet'
)
LIMIT 10;
""").fetchdf()


In [ ]:
con.execute("""SELECT
    trend_lifecycle_stage,
    COUNT(*) AS count
FROM read_parquet(
    '/content/drive/MyDrive/Capstone/reviews_lifecycle.parquet'
)
GROUP BY trend_lifecycle_stage
ORDER BY count DESC;
""").fetchdf()

In [ ]:
con.execute("DESC metadata_raw").fetchdf()

In [ ]:
con.execute("DESC reviews_raw").fetchdf()

In [ ]:
con.execute("""
COPY reviews_raw
TO '/content/drive/MyDrive/Capstone/reviews_raw.parquet'
(FORMAT PARQUET);

COPY metadata_raw
TO '/content/drive/MyDrive/Capstone/metadata_raw.parquet'
(FORMAT PARQUET);
""")

In [ ]:
con.execute("PRAGMA threads=4;")
con.execute("PRAGMA enable_progress_bar=true;")



In [ ]:
con.execute("""
DROP TABLE IF EXISTS reviews_monthly;
CREATE TABLE reviews_monthly AS
WITH base AS (
    SELECT
        parent_asin,
        DATE_TRUNC('month', to_timestamp(timestamp)) AS month_start,
        rating
    FROM reviews_raw
),

first_review AS (
    SELECT
        parent_asin,
        MIN(month_start) AS first_month
    FROM base
    GROUP BY parent_asin
)

SELECT
    b.parent_asin,

    EXTRACT(year FROM b.month_start) AS year,
    EXTRACT(month FROM b.month_start) AS month,
    STRFTIME(b.month_start, '%Y-%m') AS year_month,

    COUNT(*) AS monthly_review_count,
    AVG(b.rating) AS monthly_rating_avg,

    DATE_DIFF(
        'month',
        f.first_month,
        b.month_start
    ) AS product_age_months

FROM base b
JOIN first_review f
    ON b.parent_asin = f.parent_asin

GROUP BY
    b.parent_asin,
    year,
    month,
    year_month,
    b.month_start,
    f.first_month;
""")

In [ ]:
con.execute("SHOW TABLES").fetchdf()


In [ ]:
con.execute("""CREATE TABLE reviews_monthly AS
SELECT * FROM read_parquet('/content/drive/MyDrive/Capstone/reviews_monthly.parquet');""")


In [ ]:
con.execute("DESC reviews_monthly").fetchdf()

In [ ]:
con.close()

In [ ]:
con.execute("""
COPY reviews_monthly
TO '/content/drive/MyDrive/Capstone/reviews_monthly.parquet'
(FORMAT PARQUET);
""")
